In [20]:
import pandas as pd
import numpy as np

RAW = "../data/BigBasket_Products.csv"
CLEAN = "../data/bigbasket_clean.csv"

In [21]:
df = pd.read_csv("BigBasket_Products.csv")

In [22]:
print("Raw shape:", df.shape)
print(df.isna().sum())
raw_rows = len(df)

Raw shape: (27555, 9)
index              0
product            1
category           0
sub_category       0
brand              1
sale_price         6
market_price       0
type               0
rating          8636
dtype: int64


## RENAME COLUMNS

In [23]:
df = df.rename(columns={"index": "product_id", "type": "product_type"})

## CLEAN THE TEXT COLUMNS (EXTRA SPACES)

In [24]:
for c in ["product", "category", "sub_category", "brand", "product_type"]:
    df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

## DROP THE ROW WITH A MISSING PRODUCT NAME

In [25]:
df = df.dropna(subset=["product"])

## FILL THE MISSING BRAND

In [26]:
df["brand"] = df["brand"].fillna("Unknown")

## REMOVE DUPLICATE ROWS

In [27]:
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "product_id"])
print("Duplicates removed:", before - len(df))

Duplicates removed: 361


## FILL THE MISSING SALE PRICE

In [28]:
df["sale_price"] = df["sale_price"].fillna(df["market_price"])

## REMOVE PRICE ERRORS (SALE PRICE HIGHER THAN MRP)

In [29]:
bad = df["sale_price"] > df["market_price"]
print("Price errors removed:", bad.sum())
df = df[~bad]

Price errors removed: 5


## MISSING RATINGS: KEEP THEM EMPTY

In [30]:
print("Missing ratings kept as empty:", df["rating"].isna().sum())

Missing ratings kept as empty: 8469


## CREATE THE DISCOUNT COLUMNS

In [31]:
df["discount_amount"] = (df["market_price"] - df["sale_price"]).round(2)
df["discount_pct"] = ((df["discount_amount"] / df["market_price"]) * 100).round(2)
df["is_discounted"] = np.where(df["discount_amount"] > 0, 1, 0)

## CREATE THE PRICE BAND

In [32]:
df["price_band"] = pd.cut(
    df["sale_price"], bins=[0, 100, 500, np.inf],
    labels=["Budget (<100)", "Mid (100-500)", "Premium (>500)"], right=False
).astype("string")

## CREATE THE DISCOUNT BAND

In [33]:
df["discount_band"] = pd.cut(
    df["discount_pct"], bins=[-0.01, 0, 10, 25, 50, 100],
    labels=["No discount", "1-10%", "11-25%", "26-50%", ">50%"]
).astype("string")

## CREATE THE RATING BAND AND THE HAS_RATING FLAG

In [34]:
df["rating_band"] = pd.cut(
    df["rating"], bins=[0, 2.99, 3.99, 4.49, 5],
    labels=["Poor (<3)", "Average (3-3.9)", "Good (4-4.4)", "Excellent (4.5+)"]
).astype("string")
df["has_rating"] = df["rating"].notna().astype(int)

## CREATE THE BRAND TYPE (OWN BRAND OR THIRD-PARTY)

In [35]:
OWN = ["Fresho", "Fresho Signature", "Fresho Organic", "bb Royal", "BB Royal Organic",
       "BB Home", "BB Home Herbal", "bb Combo", "bb Popular"]
df["brand_type"] = np.where(df["brand"].isin(OWN), "BigBasket Own Brand", "Third-party Brand")

## CREATE A FRESH PRODUCT ID

In [36]:
df = df.reset_index(drop=True)
df["product_id"] = np.arange(1, len(df) + 1)

## SAVE THE CLEAN FILE

In [37]:
print(f"Rows: {raw_rows} -> {len(df)}")
df.to_csv("BigBasket_clean.csv", index=False)
print("Saved", CLEAN)

Rows: 27555 -> 27188
Saved ../data/bigbasket_clean.csv


In [38]:
check = pd.read_csv("BigBasket_clean.csv")
print(check.shape)
print(check.isna().sum()[check.isna().sum() > 0])
print(check.drop(columns="product_id").duplicated().sum())

(27188, 17)
rating         8469
rating_band    8469
dtype: int64
0
